In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
import tensorflow as tf
import numpy as np
vocab_size = 20000

(x_train, y_train), (x_test, y_test) = tf.keras.datasets.imdb.load_data(num_words=vocab_size)

In [ ]:
x_test.shape

(25000,)

In [ ]:
import numpy as np

lengths = [len(seq) for seq in x_train]

max_len = int(np.percentile(lengths, 95))

print("Chosen max_len:", max_len)

Chosen max_len: 610


In [ ]:
x_train = x_train
y_train = y_train

# x_test = x_test[:5000]
# y_test = y_test[:5000]

x_train = tf.keras.preprocessing.sequence.pad_sequences(x_train, maxlen=max_len)
x_test = tf.keras.preprocessing.sequence.pad_sequences(x_test, maxlen=max_len)

x_train = torch.tensor(x_train, dtype=torch.long)
y_train = torch.tensor(y_train, dtype=torch.long)

x_test = torch.tensor(x_test, dtype=torch.long)
y_test = torch.tensor(y_test, dtype=torch.long)
train_dataset = TensorDataset(x_train, y_train)
test_dataset = TensorDataset(x_test, y_test)

In [ ]:
x_train.shape

torch.Size([25000, 610])

In [ ]:
train_loader = DataLoader(
    train_dataset,
    batch_size=256,
    shuffle=True,
    pin_memory=True,
)

test_loader = DataLoader(
    test_dataset,
    batch_size=256,
    pin_memory=True
)

# LSTM

In [ ]:
import torch
import torch.nn as nn

class SentimentLSTM(nn.Module):

    def __init__(self, vocab_size, embed_dim, hidden_dim):

        super().__init__()

        self.embedding = nn.Embedding(vocab_size, embed_dim)

        self.lstm = nn.GRU(
            embed_dim,
            hidden_dim,
            num_layers=2,
            batch_first=True,
            bidirectional=True,
            dropout=0.6
        )

        self.fc = nn.Sequential(
            nn.Linear(hidden_dim*2, 128),
            nn.ReLU(),
            nn.Dropout(0.6),
            nn.Linear(128, 2)
        )

    def forward(self, x):

        x = self.embedding(x)

        _, (hidden, _) = self.lstm(x)

        hidden = torch.cat((hidden[-2], hidden[-1]), dim=1)

        out = self.fc(hidden)

        return out

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

model = SentimentLSTM(
    vocab_size=20000,
    embed_dim=256,
    hidden_dim=128
).to(device)

model = torch.compile(model)

cuda


In [ ]:
from adabelief_pytorch import AdaBelief

criterion = nn.CrossEntropyLoss()

optimizer = AdaBelief(model.parameters(),lr=0.0001)

Please check your arguments if you have upgraded adabelief-pytorch from version 0.0.5.
Modifications to default arguments:
                           eps  weight_decouple    rectify
-----------------------  -----  -----------------  ---------
adabelief-pytorch=0.0.5  1e-08  False              False
>=0.1.0 (Current 0.2.0)  1e-16  True               True
SGD better than Adam (e.g. CNN for Image Classification)    Adam better than SGD (e.g. Transformer, GAN)
----------------------------------------------------------  ----------------------------------------------
Recommended eps = 1e-8                                      Recommended eps = 1e-16
For a complete table of recommended hyperparameters, see
https://github.com/juntang-zhuang/Adabelief-Optimizer
You can disable the log message by setting "print_change_log = False", though it is recommended to keep as a reminder.

Weight decoupling enabled in AdaBelief
Rectification enabled in AdaBelief


In [ ]:
epochs = 20

for epoch in range(epochs):

    model.train()

    train_loss = 0
    train_correct = 0
    train_total = 0

    for x, y in train_loader:

        x = x.to(device, non_blocking=True)
        y = y.to(device, non_blocking=True)

        optimizer.zero_grad(set_to_none=True)

        outputs = model(x)

        loss = criterion(outputs, y)

        loss.backward()

        torch.nn.utils.clip_grad_norm_(model.parameters(), 1)

        optimizer.step()

        train_loss += loss.item()

        preds = torch.argmax(outputs, dim=1)

        train_correct += (preds == y).sum().item()
        train_total += y.size(0)


    train_acc = train_correct / train_total


    # TEST
    model.eval()

    test_loss = 0
    correct = 0
    total = 0

    with torch.no_grad():

        for x, y in test_loader:

            x = x.to(device, non_blocking=True)
            y = y.to(device, non_blocking=True)

            outputs = model(x)

            loss = criterion(outputs, y)

            test_loss += loss.item()

            preds = torch.argmax(outputs, dim=1)

            correct += (preds == y).sum().item()
            total += y.size(0)


    test_acc = correct / total


    print(
        f"Epoch {epoch+1} | "
        f"Train Loss {train_loss/len(train_loader):.4f} | "
        f"Train Acc {train_acc:.4f} | "
        f"Test Loss {test_loss/len(test_loader):.4f} | "
        f"Test Acc {test_acc:.4f}"
    )

Epoch 1 | Train Loss 0.6938 | Train Acc 0.4967 | Test Loss 0.6930 | Test Acc 0.5085
Epoch 2 | Train Loss 0.6932 | Train Acc 0.5026 | Test Loss 0.6922 | Test Acc 0.5538
Epoch 3 | Train Loss 0.6920 | Train Acc 0.5216 | Test Loss 0.6906 | Test Acc 0.5649
Epoch 4 | Train Loss 0.6893 | Train Acc 0.5436 | Test Loss 0.6850 | Test Acc 0.5875
Epoch 5 | Train Loss 0.6691 | Train Acc 0.5940 | Test Loss 0.6158 | Test Acc 0.6750
Epoch 6 | Train Loss 0.5626 | Train Acc 0.7216 | Test Loss 0.6788 | Test Acc 0.6146
Epoch 7 | Train Loss 0.5081 | Train Acc 0.7653 | Test Loss 0.6682 | Test Acc 0.6562
Epoch 8 | Train Loss 0.4760 | Train Acc 0.7861 | Test Loss 0.5808 | Test Acc 0.7227
Epoch 9 | Train Loss 0.4515 | Train Acc 0.8025 | Test Loss 0.7340 | Test Acc 0.6560
Epoch 10 | Train Loss 0.4314 | Train Acc 0.8120 | Test Loss 0.6854 | Test Acc 0.7001
Epoch 11 | Train Loss 0.3989 | Train Acc 0.8294 | Test Loss 0.6579 | Test Acc 0.7390
Epoch 12 | Train Loss 0.3740 | Train Acc 0.8440 | Test Loss 0.6808 | Test 